# 01 — Generate NFL Combine Data

Creates ~300 rows of realistic fake NFL Combine results using `faker` and writes
the DataFrame to `alexander_booth.nfl_combine.combine_results` as a managed Delta table.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "nfl_combine")
TABLE_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.combine_results"

In [2]:
import random
from faker import Faker
import pandas as pd

fake = Faker()
Faker.seed(42)
random.seed(42)

POSITIONS = ["QB", "RB", "WR", "TE", "OT", "OG", "C", "DE", "DT", "LB", "CB", "S", "K", "P"]

COLLEGES = [
    "Alabama", "Ohio State", "Georgia", "LSU", "Clemson", "Michigan",
    "Oklahoma", "Texas", "USC", "Oregon", "Penn State", "Notre Dame",
    "Florida", "Tennessee", "Texas A&M", "Miami", "Auburn", "Wisconsin",
    "Iowa", "Stanford", "UCLA", "Florida State", "Ole Miss", "Arkansas",
    "Pittsburgh", "North Carolina", "Baylor", "Kentucky", "Utah",
    "Colorado", "Washington", "BYU", "Cincinnati", "Tulane",
]

# Position-based physical profiles (height_in, weight_lbs ranges)
POSITION_PROFILES = {
    "QB":  (73, 77, 210, 240), "RB":  (68, 72, 195, 230), "WR":  (70, 76, 175, 215),
    "TE":  (74, 78, 240, 265), "OT":  (76, 80, 295, 340), "OG":  (74, 78, 295, 330),
    "C":   (73, 77, 290, 315), "DE":  (74, 78, 250, 280), "DT":  (73, 77, 290, 340),
    "LB":  (72, 76, 225, 260), "CB":  (69, 73, 180, 205), "S":   (70, 74, 195, 215),
    "K":   (70, 74, 180, 210), "P":   (72, 76, 200, 225),
}

DRAFT_YEARS = [2023, 2024, 2025]

rows = []
for _ in range(300):
    pos = random.choice(POSITIONS)
    h_lo, h_hi, w_lo, w_hi = POSITION_PROFILES[pos]
    rows.append({
        "player_name":      fake.name_male(),
        "position":         pos,
        "college":          random.choice(COLLEGES),
        "height_in":        random.randint(h_lo, h_hi),
        "weight_lbs":       random.randint(w_lo, w_hi),
        "forty_yard_dash":  round(random.uniform(4.3, 5.4), 2),
        "bench_press_reps": random.randint(10, 35),
        "vertical_jump_in": round(random.uniform(25.0, 42.0), 1),
        "broad_jump_in":    random.randint(108, 132),
        "three_cone_drill": round(random.uniform(6.5, 7.8), 2),
        "shuttle_20yd":     round(random.uniform(3.9, 4.6), 2),
        "draft_year":       random.choice(DRAFT_YEARS),
    })

pdf = pd.DataFrame(rows)
print(f"Generated {len(pdf)} combine records")
pdf.head(10)

Generated 300 combine records


,player_name,position,college,height_in,weight_lbs,forty_yard_dash,bench_press_reps,vertical_jump_in,broad_jump_in,three_cone_drill,shuttle_20yd,draft_year
0,Alexander Hill,CB,Texas,69,203,4.60,17,27.4,111,7.38,4.52,2023
1,Noah Rhodes,LB,Kentucky,72,226,4.40,17,33.6,108,7.23,4.40,2025
2,Brandon Henderson,DT,Baylor,74,318,4.95,35,39.8,132,7.55,4.39,2024
3,Daniel Wagner,OT,Oregon,77,316,4.41,22,26.6,119,7.28,4.46,2025
4,Cristian Santos,DE,Texas,77,252,4.91,30,35.5,119,7.25,4.39,2023
5,Daniel Lawrence,CB,Texas A&M,71,182,5.24,13,31.5,122,7.33,4.16,2024
6,Aaron Shaffer,OG,Tennessee,76,299,4.97,15,34.1,115,6.71,4.17,2025
7,Gerald Moore,S,Texas A&M,72,196,4.55,11,38.7,120,6.85,4.05,2025
8,George Davis,S,UCLA,71,215,4.85,30,32.8,116,6.68,4.42,2025
9,Ryan Munoz,OT,Kentucky,80,320,4.70,14,33.7,110,7.48,4.50,2023


In [3]:
# Write to Delta via Databricks Connect
sdf = spark.createDataFrame(pdf)

sdf.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(TABLE_NAME)

print(f"Wrote {sdf.count()} rows to {TABLE_NAME}")

Wrote 300 rows to alexander_booth.nfl_combine.combine_results


In [4]:
# Quick sanity check
spark.sql(f"SELECT position, COUNT(*) AS cnt, ROUND(AVG(forty_yard_dash),2) AS avg_40 FROM {TABLE_NAME} GROUP BY position ORDER BY avg_40").show(20)

+--------+---+------+
|position|cnt|avg_40|
+--------+---+------+
|      OG| 20|  4.75|
|       P| 30|  4.76|
|      CB| 21|  4.78|
|      WR| 24|  4.79|
|      TE| 13|  4.79|
|       C| 23|   4.8|
|      LB| 33|  4.82|
|       S| 20|  4.86|
|      OT| 23|  4.86|
|      DE| 23|  4.87|
|      RB| 15|  4.89|
|       K| 21|  4.89|
|      QB| 21|   4.9|
|      DT| 13|  4.96|
+--------+---+------+

